<a href="https://colab.research.google.com/github/ShakilCuambe/Gera-o-de-imagens---stable-diffusion/blob/main/is_deep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import zipfile
from diffusers import StableDiffusionPipeline
from peft import LoraConfig, get_peft_model
import torch
import numpy as np
from PIL import Image
import os


"""
drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/cats.zip"
extract_path = "/content/cats"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extraído!")
"""

device = "cuda" if torch.cuda.is_available() else "cpu"

# Carregar modelo
model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_id).to(device)

# Configuração LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["to_q", "to_k", "to_v"],
    lora_dropout=0.1,
)
# Modelagem
pipe.unet = get_peft_model(pipe.unet, lora_config)

# Dataset
image_folder = "cats"
images = []

for f in os.listdir(image_folder):
    if f.lower().endswith((".jpg", ".png", ".jpeg")):
        img = Image.open(os.path.join(image_folder, f)).convert("RGB")
        images.append(img)

# Otimizador
optimizer = torch.optim.Adam(pipe.unet.parameters(), lr=1e-4)

# Treino
for epoch in range(3):
    for img in images:
        img = img.resize((512, 512))

        # Tenosres
        img_array = np.array(img)
        img_tensor = torch.tensor(img_array).float().permute(2, 0, 1).unsqueeze(0).to(device) / 255.0


        with torch.no_grad():
            latents = pipe.vae.encode(img_tensor).latent_dist.sample()

        latents = latents * 0.18215

        # Passar pelo UNet
        output = pipe.unet(
            latents,
            timestep=torch.tensor([0]).to(device),
            encoder_hidden_states=torch.zeros((1, 77, 768)).to(device)
        ).sample

        loss = output.mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch} concluído")

# Gerar imagem
pipe = StableDiffusionPipeline.from_pretrained(model_id).to(device)

prompt = "a cute fluffy cat, ultra realistic, soft lighting"

image = pipe(prompt).images[0]
image.save("gato.png")

print("Imagem gerada com suxexooo")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FileNotFoundError: [Errno 2] No such file or directory: 'cats'

In [1]:

from diffusers import StableDiffusionPipeline
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)

pipe = pipe.to(device)
pipe.safety_checker = None

pipe.enable_attention_slicing()

# prompt = "anime girl, blue eyes, soft lighting, high quality, full body"


base = "capitalist leader"

variacoes = [
    "smiling",
    "serious",
    "looking left",
    "portrait",
    "close up",
    "running",
    "climbing",
    "hugging",
    "dying"
]

negative = "blurry, low quality, distorted data"

for i, v in enumerate(variacoes):
    prompt = f"{base}, {v}"

    generator = torch.manual_seed(42)

    image = pipe(
        prompt,
        negative_prompt=negative,
        num_inference_steps=50,
        guidance_scale=7.5,
        generator=generator
    ).images[0]

    image.save(f"personagem{i}.png")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

...

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print(torch.cuda.is_available())